# L13 – Lösningar

In [ ]:
import sys
sys.path.insert(0, "..")
import math, re
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from datasets import load_dataset
from dlkit.seed import set_seed
from dlkit.train import fit

device = "cuda" if torch.cuda.is_available() else "cpu"

### Övning 1

In [ ]:
def my_attention(Q, K, V):
    d = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d)    # (N, L, L)
    weights = torch.softmax(scores, dim=-1)
    return weights @ V                                 # (N, L, d)

set_seed(0)
Qt, Kt, Vt = torch.randn(2, 5, 8), torch.randn(2, 5, 8), torch.randn(2, 5, 8)
assert torch.allclose(my_attention(Qt, Kt, Vt), F.scaled_dot_product_attention(Qt, Kt, Vt), atol=1e-6)
print("Rätt!")

### Övning 2

In [ ]:
class NoPosTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=64, nhead=4, num_layers=2):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        layer = nn.TransformerEncoderLayer(d_model, nhead, 4 * d_model, dropout=0.1,
                                           activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers, enable_nested_tensor=False)
        self.head = nn.Linear(d_model, 2)

    def forward(self, x):
        pad = x == 0
        h = self.encoder(self.tok_emb(x), src_key_padding_mask=pad)
        keep = (~pad).unsqueeze(-1).float()
        return self.head((h * keep).sum(1) / keep.sum(1).clamp(min=1))

set_seed(0)
m = NoPosTransformer(1000).eval()                 # eval: dropout av, så att jämförelsen blir deterministisk
x = torch.randint(2, 1000, (1, 20))               # 20 riktiga tokens
x_shuffled = x[:, torch.randperm(20)]
with torch.no_grad():
    same_output = bool(torch.allclose(m(x), m(x_shuffled), atol=1e-5))
assert same_output is True
print("Rätt! Utan position ser modellen bara en 'påse' med ord.")

**Svar:** utan positionsembedding är modellen en avancerad bag of words, precis som TF-IDF. För sentiment räcker det långt, eftersom vilka ord som förekommer säger mycket. Det misslyckas dock på negationer som "not good" kontra "good, not bad". För översättning, frågor och svar eller kod är ordningen avgörande. "Hunden bet mannen" och "mannen bet hunden" ska ge olika översättningar.

### Övning 3

In [ ]:
imdb = load_dataset("stanfordnlp/imdb", cache_dir="../data/hf")
train_all = imdb["train"].shuffle(seed=42)
train_texts, train_labels = train_all["text"][:20_000], train_all["label"][:20_000]
val_texts, val_labels = train_all["text"][20_000:], train_all["label"][20_000:]

def tokenize(text):
    return re.findall(r"[a-z0-9']+", text.lower().replace("<br />", " "))

counts = Counter(t for text in train_texts for t in tokenize(text))
itos = ["<pad>", "<unk>"] + [w for w, _ in counts.most_common(20_000)]
stoi = {w: i for i, w in enumerate(itos)}
MAX_LEN = 256
encode = lambda text: ([stoi.get(t, 1) for t in tokenize(text)][:MAX_LEN] + [0] * MAX_LEN)[:MAX_LEN]
to_loader = lambda texts, labels, bs, sh: DataLoader(
    TensorDataset(torch.tensor([encode(t) for t in texts]), torch.tensor(labels)), batch_size=bs, shuffle=sh)
train_loader = to_loader(train_texts, train_labels, 64, True)
val_loader = to_loader(val_texts, val_labels, 256, False)

class TextTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=64, nhead=4, num_layers=2, max_len=MAX_LEN, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)
        layer = nn.TransformerEncoderLayer(d_model, nhead, 4 * d_model, dropout=dropout,
                                           activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers, enable_nested_tensor=False)
        self.head = nn.Linear(d_model, 2)

    def forward(self, x):
        pad = x == 0
        h = self.tok_emb(x) + self.pos_emb(torch.arange(x.size(1), device=x.device))
        h = self.encoder(h, src_key_padding_mask=pad)
        keep = (~pad).unsqueeze(-1).float()
        return self.head((h * keep).sum(1) / keep.sum(1).clamp(min=1))

set_seed(42)
big = TextTransformer(len(itos), d_model=128, num_layers=4).to(device)
print(f"{sum(p.numel() for p in big.parameters()):,} parametrar")
h_big = fit(big, train_loader, val_loader, nn.CrossEntropyLoss(),
            torch.optim.AdamW(big.parameters(), lr=5e-4, weight_decay=0.01), device, epochs=8, patience=2)

**Tolkning:** den större modellen blir sällan bättre här. Train accuracy stiger snabbare medan val loss vänder uppåt tidigare. Det är **överanpassning** (L06): datan räcker inte för fler parametrar. Mer kapacitet hjälper bara om modellen *under*anpassar. Här är det mer data (eller förträning) som behövs.